In [1]:
import os
import pandas as pd
from datetime import datetime
import pytz
import unicodedata

# === Date & Paths ===
eastern = pytz.timezone("US/Eastern")
today = datetime.now(eastern).date()
today_str = today.isoformat()

paths = {
    "spreads": f"../data/novig-odds/novig_spreads_{today_str}.csv",
    "pitching_stats": f"../data/pitching_stats/pitching_stats_{today_str}.csv",
    "standings": f"../data/standings/standings_{today_str}.csv",
    "team_batting": f"../data/team_batting/team_batting_{today_str}.csv",
    "team_pitching": f"../data/team_pitching/team_pitching_{today_str}.csv"
}

# === Load Data ===
for label, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required file: {path}")

spreads_df = pd.read_csv(paths["spreads"])
pitching_stats_df = pd.read_csv(paths["pitching_stats"])
standings_df = pd.read_csv(paths["standings"])
batting_df = pd.read_csv(paths["team_batting"])
teamp_df = pd.read_csv(paths["team_pitching"])

# === Normalize ===
def normalize_str(s):
    if not isinstance(s, str): return ""
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8').strip().lower()

# === Preprocess Spreads ===
spreads_df['fav_score'] = None
spreads_df['dog_score'] = None
spreads_df['home_score'] = None
spreads_df['away_score'] = None

# === Normalize team names ===
pitching_stats_df['Home Team Clean'] = pitching_stats_df['Home Team'].apply(normalize_str)
pitching_stats_df['Away Team Clean'] = pitching_stats_df['Away Team'].apply(normalize_str)

# === Map team abbreviations to full names ===
team_name_map = {
    "ATL": "Atlanta Braves", "PHI": "Philadelphia Phillies", "OAK": "Athletics",
    "TOR": "Toronto Blue Jays", "TB": "Tampa Bay Rays", "HOU": "Houston Astros",
    "WAS": "Washington Nationals", "SEA": "Seattle Mariners", "CIN": "Cincinnati Reds",
    "CHC": "Chicago Cubs", "MIL": "Milwaukee Brewers", "CWS": "Chicago White Sox",
    "BAL": "Baltimore Orioles", "LAA": "Los Angeles Angels", "CLE": "Cleveland Guardians",
    "COL": "Colorado Rockies", "NYM": "New York Mets", "SF": "San Francisco Giants",
    "MIA": "Miami Marlins", "BOS": "Boston Red Sox", "STL": "St. Louis Cardinals",
    "TEX": "Texas Rangers", "DET": "Detroit Tigers", "KC": "Kansas City Royals",
    "KAN": "Kansas City Royals", "AZ": "Arizona Diamondbacks", "ARI": "Arizona Diamondbacks",
    "PIT": "Pittsburgh Pirates", "SD": "San Diego Padres", "MIN": "Minnesota Twins",
    "NYY": "New York Yankees", "LAD": "Los Angeles Dodgers"
}

# Derive home and away teams
spreads_df['home_team'] = spreads_df.apply(lambda row: row['fav_team'] if row['home_favorite'] == 1 else row['dog_team'], axis=1)
spreads_df['away_team'] = spreads_df.apply(lambda row: row['dog_team'] if row['home_favorite'] == 1 else row['fav_team'], axis=1)
spreads_df['home_team_full'] = spreads_df['home_team'].map(lambda abbr: team_name_map.get(abbr.upper(), ""))
spreads_df['away_team_full'] = spreads_df['away_team'].map(lambda abbr: team_name_map.get(abbr.upper(), ""))
spreads_df['home_team_clean'] = spreads_df['home_team_full'].apply(normalize_str)
spreads_df['away_team_clean'] = spreads_df['away_team_full'].apply(normalize_str)

# === Assign Pitchers and Stats ===
output_rows = []

for _, row in spreads_df.iterrows():
    home = row['home_team_clean']
    away = row['away_team_clean']

    ps_row = pitching_stats_df[
        (pitching_stats_df['Home Team Clean'] == home) &
        (pitching_stats_df['Away Team Clean'] == away)
    ]

    if ps_row.empty:
        ps_row = pitching_stats_df[
            (pitching_stats_df['Home Team Clean'] == away) &
            (pitching_stats_df['Away Team Clean'] == home)
        ]
        flipped = not ps_row.empty
    else:
        flipped = False

    if not ps_row.empty:
        ps_row = ps_row.iloc[0]
        if flipped:
            home_pitcher, away_pitcher = ps_row['Away Starter'], ps_row['Home Starter']
            home_stats = {
                'era': ps_row['Away ERA'], 'whip': ps_row['Away WHIP'], 'so': ps_row['Away SO'],
                'ip': ps_row['Away IP'], 'avg': ps_row['Away AVG']
            }
            away_stats = {
                'era': ps_row['Home ERA'], 'whip': ps_row['Home WHIP'], 'so': ps_row['Home SO'],
                'ip': ps_row['Home IP'], 'avg': ps_row['Home AVG']
            }
        else:
            home_pitcher, away_pitcher = ps_row['Home Starter'], ps_row['Away Starter']
            home_stats = {
                'era': ps_row['Home ERA'], 'whip': ps_row['Home WHIP'], 'so': ps_row['Home SO'],
                'ip': ps_row['Home IP'], 'avg': ps_row['Home AVG']
            }
            away_stats = {
                'era': ps_row['Away ERA'], 'whip': ps_row['Away WHIP'], 'so': ps_row['Away SO'],
                'ip': ps_row['Away IP'], 'avg': ps_row['Away AVG']
            }

        if row['home_favorite'] == 1:
            fav_pitcher, dog_pitcher = home_pitcher, away_pitcher
            fav_stats, dog_stats = home_stats, away_stats
        else:
            fav_pitcher, dog_pitcher = away_pitcher, home_pitcher
            fav_stats, dog_stats = away_stats, home_stats
    else:
        fav_pitcher = dog_pitcher = "Unknown"
        fav_stats = dog_stats = {k: None for k in ['era', 'whip', 'so', 'ip', 'avg']}

    row = row.copy()
    row['fav_pitcher'] = fav_pitcher
    row['dog_pitcher'] = dog_pitcher
    for k in fav_stats:
        row[f'fav_pitcher_{k}'] = fav_stats[k]
        row[f'dog_pitcher_{k}'] = dog_stats[k]

    output_rows.append(row)

merged_df = pd.DataFrame(output_rows)

# === Normalize and Join Team Stats ===
for df in [standings_df, batting_df, teamp_df]:
    df['team_clean'] = df['Tm'].apply(normalize_str)

def get_team_stat(team_abbr, df, column):
    full = team_name_map.get(team_abbr.upper(), "")
    clean = normalize_str(full)
    row = df[df['team_clean'] == clean]
    return row[column].values[0] if not row.empty else None

# Standings
merged_df['fav_win_pct'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, 'W-L%'))
merged_df['dog_win_pct'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, 'W-L%'))

# Batting/ERA
merged_df['fav_ba'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, 'BA'))
merged_df['dog_ba'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, 'BA'))
merged_df['fav_era_p'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, 'ERA'))
merged_df['dog_era_p'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, 'ERA'))

# Additional standings columns
standings_columns = ['W','L','Strk','R','RA','SOS','SRS','pythWL','Luck','last10','last20','last30']
for col in standings_columns:
    merged_df[f'fav_{col}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, col))
    merged_df[f'dog_{col}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, col))

# Additional batting columns
batting_columns = ['R/G','PA','AB','R','H','2B','3B','HR','RBI','SB','BB','SO','BA','OBP','SLG','OPS','OPS+']
for col in batting_columns:
    col_renamed = f'{col}_b' if col in ['R','H','BB','SO'] else col
    merged_df[f'fav_{col_renamed}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, col))
    merged_df[f'dog_{col_renamed}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, col))

# Additional pitching columns
pitching_columns = ['RA/G','ERA','SV','H','R','ER','HR','BB','SO','ERA+','FIP','WHIP','H9']
for col in pitching_columns:
    col_renamed = f'{col}_p' if col in ['H','R','BB','SO'] else col
    merged_df[f'fav_{col_renamed}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, col))
    merged_df[f'dog_{col_renamed}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, col))

# === Final Ordering ===
first_cols = [
    'game_time_est', 'fav_team', 'fav_score', 'dog_team', 'dog_score',
    'home_team', 'home_score', 'away_team', 'away_score',
    'fav_line', 'dog_line', 'fav_price', 'dog_price',
    'fav_price_american', 'dog_price_american', 'home_favorite', 'market',
    'fav_pitcher', 'dog_pitcher',
    'fav_pitcher_era', 'dog_pitcher_era',
    'fav_pitcher_whip', 'dog_pitcher_whip',
    'fav_pitcher_so', 'dog_pitcher_so',
    'fav_pitcher_ip', 'dog_pitcher_ip',
    'fav_pitcher_avg', 'dog_pitcher_avg',
    'fav_win_pct', 'dog_win_pct', 'fav_ba', 'dog_ba', 'fav_era_p', 'dog_era_p'
]
remaining_cols = [col for col in merged_df.columns if col not in first_cols and col not in ['market_timestamp', 'home_team_full', 'away_team_full', 'home_team_clean', 'away_team_clean']]
merged_df = merged_df[first_cols + remaining_cols]

# === Output ===
print("\u2705 Final enriched training set (WAR excluded):")
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
display(merged_df)

# === Save CSV ===
output_path = f"../training-data/training-set/training_set_{today_str}.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
merged_df.to_csv(output_path, index=False)
print(f"\u2705 Enriched training set saved to {output_path}")


✅ Final enriched training set (WAR excluded):


,game_time_est,fav_team,fav_score,dog_team,dog_score,home_team,home_score,away_team,away_score,fav_line,dog_line,fav_price,dog_price,fav_price_american,dog_price_american,home_favorite,market,fav_pitcher,dog_pitcher,fav_pitcher_era,dog_pitcher_era,fav_pitcher_whip,dog_pitcher_whip,fav_pitcher_so,dog_pitcher_so,fav_pitcher_ip,dog_pitcher_ip,fav_pitcher_avg,dog_pitcher_avg,fav_win_pct,dog_win_pct,fav_ba,dog_ba,fav_era_p,dog_era_p,fav_W,dog_W,fav_L,dog_L,fav_Strk,dog_Strk,fav_R,dog_R,fav_RA,dog_RA,fav_SOS,dog_SOS,fav_SRS,dog_SRS,fav_pythWL,dog_pythWL,fav_Luck,dog_Luck,fav_last10,dog_last10,fav_last20,dog_last20,fav_last30,dog_last30,fav_R/G,dog_R/G,fav_PA,dog_PA,fav_AB,dog_AB,fav_R_b,dog_R_b,fav_H_b,dog_H_b,fav_2B,dog_2B,fav_3B,dog_3B,fav_HR,dog_HR,fav_RBI,dog_RBI,fav_SB,dog_SB,fav_BB_b,dog_BB_b,fav_SO_b,dog_SO_b,fav_BA,dog_BA,fav_OBP,dog_OBP,fav_SLG,dog_SLG,fav_OPS,dog_OPS,fav_OPS+,dog_OPS+,fav_RA/G,dog_RA/G,fav_ERA,dog_ERA,fav_SV,dog_SV,fav_H_p,dog_H_p,fav_R_p,dog_R_p,fav_ER,dog_ER,fav_BB_p,dog_BB_p,fav_SO_p,dog_SO_p,fav_ERA+,dog_ERA+,fav_FIP,dog_FIP,fav_WHIP,dog_WHIP,fav_H9,dog_H9
0,2025-06-09T21:38:00-04:00,LAA,None,OAK,None,OAK,None,LAA,None,-1.5,1.5,0.412,0.592,143,-145,0,LAA -1.5,Yusei Kikuchi,TBD,3.23,NaN,1.59,NaN,63.0,NaN,69.2,NaN,0.266,NaN,0.469,0.388,.225,.257,4.85,5.68,30,26,34,41,L 1,W 1,4.2,4.3,5.1,6.0,0.0,0.0,-0.8,-1.8,27-37,24-43,3.0,2.0,5-5,3-7,11-9,4-16,16-14,6-24,4.22,4.31,2340,2554,2126,2299,270,289,479,591,91,114,7,8,82,101,260,280,28,37,163,220,622,551,.225,.257,.287,.324,.406,.418,.693,.742,94,109,5.08,6.03,4.85,5.68,17,15,592,646,325,404,303,375,263,267,475,532,86,73,4.79,5.01,1.520,1.536,9.5,9.8
1,2025-06-09T19:45:00-04:00,STL,None,TOR,None,TOR,None,STL,None,-1.5,1.5,0.360,0.669,178,-202,0,STL -1.5,Andre Pallante,José Berríos,4.91,3.67,1.38,1.28,44.0,70.0,66.0,76.0,0.262,0.238,0.554,0.538,.258,.252,3.87,4.02,36,35,29,30,L 1,L 1,4.6,4.2,4.2,4.3,0.1,0.0,0.6,0.0,35-30,32-33,1.0,3.0,5-5,8-2,11-9,13-7,20-10,19-11,4.63,4.25,2473,2461,2204,2185,301,276,569,550,119,112,4,3,51,90,286,266,37,37,210,219,506,446,.258,.252,.329,.324,.395,.395,.723,.718,103,101,4.20,4.26,3.87,4.02,17,20,548,513,273,277,248,259,180,195,464,588,105,104,3.65,4.21,1.262,1.221,8.6,8.0
2,2025-06-09T18:40:00-04:00,CLE,None,CIN,None,CLE,None,CIN,None,-1.5,1.5,0.422,0.600,137,-150,1,CLE -1.5,Luis L. Ortiz,Wade Miley,4.02,18.00,1.40,3.00,71.0,0.0,65.0,2.0,0.237,0.500,0.531,0.500,.234,.244,3.95,3.71,34,33,30,33,W 1,W 3,3.9,4.6,4.2,4.0,0.1,-0.1,-0.2,0.5,30-34,37-29,4.0,-4.0,5-5,5-5,9-11,11-9,14-16,15-15,3.91,4.56,2350,2480,2090,2208,250,301,489,539,89,111,6,7,69,75,236,289,50,57,203,232,523,582,.234,.244,.306,.320,.377,.397,.683,.717,93,95,4.20,3.98,3.95,3.71,19,18,537,499,269,263,248,241,234,196,543,529,103,119,4.07,4.05,1.365,1.188,8.6,7.7
3,2025-06-09T18:45:00-04:00,PHI,None,CHC,None,PHI,None,CHC,None,-1.5,1.5,0.366,0.668,173,-201,1,PHI -1.5,Zack Wheeler,Matthew Boyd,2.96,3.01,0.92,1.19,94.0,70.0,76.0,68.2,0.190,0.256,0.569,0.615,.250,.259,4.01,3.67,37,40,28,25,L 5,L 1,4.6,5.6,4.3,4.0,-0.3,-0.1,0.0,1.5,34-31,42-23,3.0,-2.0,1-9,6-4,10-10,14-6,17-13,19-11,4.57,5.58,2482,2539,2209,2257,297,363,552,585,100,121,9,15,66,70,281,355,58,82,231,237,514,514,.250,.259,.325,.331,.396,.443,.721,.774,100,122,4.34,3.97,4.01,3.67,20,15,562,530,282,258,259,237,199,184,611,506,104,104,3.57,3.95,1.310,1.230,8.7,8.2
4,2025-06-09T21:40:00-04:00,LAD,None,SD,None,SD,None,LAD,None,-1.5,1.5,0.410,0.626,144,-167,0,SD +1.5,Dustin May,Nick Pivetta,4.09,3.16,1.25,1.02,63.0,76.0,61.2,68.1,0.236,0.204,0.591,0.578,.266,.246,4.11,3.42,39,37,27,27,W 1,W 1,5.6,4.0,4.4,3.8,0.0,-0.1,1.1,0.2,40-26,34-30,-1.0,3.0,5-5,6-4,10-10,10-10,15-15,14-16,5.56,4.05,2559,2373,2264,2127,367,259,602,524,113,90,11,8,85,63,351,239,42,49,246,196,545,455,.266,.246,.341,.312,.461,.375,.802,.687,127,92,4.44,3.77,4.11,3.42,19,24,537,471,293,241,269,216,231,213,595,553,95,118,4.24,3.82,1.304,1.204,8.2,7.5
5,2025-06-09T21:40:00-04:00,ARI,None,SEA,None,ARI,None,SEA,None,-1.5,1.5,0.449,0.581,123,-139,1,ARI -

✅ Enriched training set saved to ../training-data/training-set/training_set_2025-06-09.csv
